## 1. Scraping URL's of bikes 

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import pandas as pd

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

cities = [
    "delhi", "mumbai", "bangalore", "chennai", "hyderabad", "kolkata",
    "jaipur", "pune", "ahmedabad", "surat", "chandigarh",
    "lucknow", "nagpur", "bhopal", "indore", "coimbatore"
]

bike_urls = set()

for loc in cities:
    print(f"\nScraping {loc}...")
    driver.get(f'https://www.bikewale.com/used/bikes-in-{loc}/')
    time.sleep(3)

    while True:
        last_height = driver.execute_script("return document.body.scrollHeight")
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == last_height:
            print(f"Fully loaded {loc}")
            break

        last_height = new_height

    before = len(bike_urls)
    soup = BeautifulSoup(driver.page_source, 'lxml')
    for a in soup.find_all("a", href=True):
        href = a['href']
        if "/used/" in href and f"bikes-in-{loc}" in href and href.endswith("/") and len(href.split("/")) >= 4:
            bike_urls.add("https://www.bikewale.com" + href)

    after = len(bike_urls)
    print(f"{loc}: {after - before} bikes found")

pd.DataFrame(list(bike_urls), columns=['url']).to_csv('bike_urls.csv', index=False)
print(f"\nTotal URLs saved: {len(bike_urls)}")
driver.quit()

## 2. Scraping bike data 

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import pandas as pd
import time

def safe_get(lst, index, default='Not Available'):
    try:
        return lst[index].text.strip()
    except IndexError:
        return default

def get_spec_by_label(soup, label):
    try:
        label_tag = soup.find(lambda tag: tag.text.strip() == label)
        if label_tag:
            value = label_tag.find_next("div", {'class': 'o-jJ o-jq o-c8 o-j3 o-bQ o-cm'})
            if value:
                return value.text.strip()
        return 'Not Available'
    except:
        return 'Not Available'

url_df = pd.read_csv('bike_urls.csv')
url_list = url_df['url'].tolist()

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

all_bikes = []

for i, url in enumerate(url_list):
    try:
        driver.get(url)
        time.sleep(3)

        soup = BeautifulSoup(driver.page_source, 'lxml')

        details = soup.find_all("div", {'class': 'o-f5 o-eS o-js o-j3'})

        bike = {
            'title'                  : soup.find("h1", {'class': 'o-j7 o-jJ o-js'}).text.strip(),
            'price'                  : safe_get(details, 0),
            'kilometer'              : safe_get(details, 1),
            'fuel_type'              : safe_get(details, 2),
            'registration_type'      : safe_get(details, 3),
            'registration_year'      : safe_get(details, 4),
            'owner'                  : safe_get(details, 5),
            'color'                  : safe_get(details, 6),
            'city'                   : safe_get(details, 7),
            'insurance'              : safe_get(details, 8),
            'last_updated'           : safe_get(details, 10),
            'mileage_arai'           : get_spec_by_label(soup, 'Mileage - ARAI'),
            'mileage_owner_reported' : get_spec_by_label(soup, 'Mileage - Owner Reported'),
            'riding_range'           : get_spec_by_label(soup, 'Riding Range'),
            'cylinders'              : get_spec_by_label(soup, 'Cylinders'),
            'front_brake_type'       : get_spec_by_label(soup, 'Front Brake Type'),
            'rear_brake_type'        : get_spec_by_label(soup, 'Rear Brake Type'),
            'tyre_type'              : get_spec_by_label(soup, 'Tyre Type'),
            'fuel_tank_capacity'     : get_spec_by_label(soup, 'Fuel Tank Capacity'),
            'reserve_fuel_capacity'  : get_spec_by_label(soup, 'Reserve Fuel Capacity'),
        }

        all_bikes.append(bike)

        if i % 100 == 0:
            pd.DataFrame(all_bikes).to_csv('bikes_data.csv', index=False)
            print(f"Scraped {i}/{len(url_list)} bikes")

    except Exception as e:
        print(f"Error at bike {i} → {url}: {e}")
        continue

bike_df = pd.DataFrame(all_bikes)
bike_df.to_csv('bikes_data.csv', index=False)
print(f"\nTotal bikes scraped: {len(all_bikes)}")
print(bike_df.head())

driver.quit()